# 2026 Agentic AI Exposure Classification Pipeline

**Purpose**: Classify O\*NET task statements against the 2026 Agentic AI Exposure Rubric
using Google Gemini via Vertex AI.

**Model**: `gemini-3.1-pro-preview` · Vertex AI · Application Default Credentials (Launched Feb 2026)

**Output**: Each task receives an exposure label (E0 / E1 / E2 / E3) with a concise explanation.

**Reproducibility**: Deterministic settings (temperature 0.1, top_p 0.95), incremental saves,
resume-safe, structured JSON enforcement with automatic retry.

## 1 · Environment Setup & Configuration

In [12]:
# ── Imports ──────────────────────────────────────────────────────────
import json
import time
import logging
import pathlib
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone

import pandas as pd
from tqdm.notebook import tqdm

from google import genai
from google.genai import types

# ── Configurable Parameters ──────────────────────────────────────────
PROJECT_ID       = "eloundou-new-scores"   # ← replace with your project
LOCATION         = "global"
MODEL_NAME       = "gemini-3.1-pro-preview"

INPUT_FILE       = pathlib.Path("data/Task Statements.xlsx")
OUTPUT_FILE      = pathlib.Path("data/Task_Statements_Classified_04.xlsx")
CHECKPOINT_FILE  = pathlib.Path("data/_checkpoint_exposure_04.parquet")

BATCH_SIZE            = 10    # Tasks per API call
PARALLEL_BATCH_WORKERS = 2   # Concurrent batches in flight
MAX_TASKS             = None   # None = full run
FRESH_START           = False
DRY_RUN               = False
SAVE_EVERY_N          = 10   # Checkpoint every N completed tasks
SLEEP_SECONDS         = 0    # Rely on backoff/cooldown instead

TEMPERATURE       = 0.1
MAX_OUTPUT_TOKENS = 15000     # Sufficient for BATCH_SIZE=10 JSON output
TOP_P             = 0.95

# ── Logging ──────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("exposure")
log.info("Configuration loaded.")

# ── Initialize GenAI Client ──────────────────────────────────────────
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
log.info(f"GenAI client ready  project={PROJECT_ID}  location={LOCATION}  model={MODEL_NAME}")

13:24:35  INFO      Configuration loaded.
13:24:35  INFO      GenAI client ready  project=eloundou-new-scores  location=global  model=gemini-3.1-pro-preview


## 2 · Exposure Rubric & Cached Content

The full 2026 Agentic AI Exposure Rubric is cached via `client.caches.create()`
so it is uploaded **once** and reused across all requests (reducing prompt tokens
and throttling pressure). If caching is unavailable, the rubric is sent inline as before.

In [13]:
RUBRIC = """\
E Exposure Taxonomy
Consider the most capable LLM models available in 2026 in an agentic chatbot form. This system combines a frontier LLM with built in tool use, meaning it can make multi step plans, call tools, and take actions inside a web browser as well as inside common digital workspaces such as Microsoft 365 and Google Workspace when the worker has access and permissions set up. The system can navigate websites, fill forms, draft and edit documents, manipulate spreadsheets, create presentations, manage email and calendars, and execute structured workflows within these general purpose tools. Accessing internal systems through standard browser interaction using the worker's credentials counts as allowed capability, provided no specialized API level integration or custom enterprise tooling is required.
The system can retrieve up to date facts from the public internet via browsing, but it still depends on what it is allowed to access. It does not have access to private systems without credentials, proprietary databases unless provided, physical world sensors or actuators, or embedded enterprise back end integrations unless explicitly assumed.
Assume you are a worker with an average level of expertise in your role trying to complete the given task. You have access to this agentic chatbot and any other existing software or computer hardware tools mentioned in the task. You also have access to any commonly available technical tools accessible via a laptop, for example a microphone, speakers, a camera, standard office software, and common web services. You do not have access to any other physical tools or materials beyond what the task explicitly provides. Please label the given task according to the taxonomy below.
When determining exposure, apply a counterfactual test: can access to the described agentic system reduce the time required to complete the task at equivalent quality by at least half? Focus on the primary bottleneck of the task rather than marginal improvements in small subcomponents.

E0: No exposure -> Label tasks E0 if direct access to the 2026 agentic chatbot interface, cannot reduce the time it takes to complete this task with equivalent quality by half or more.
Tasks should also be classified as E0 if more than half of the work is inherently physical, embodied, or depends on in person human interaction that cannot be substituted by text, workspace actions, or browser based actions. Examples include operating machinery, performing a physical inspection or repair that requires touch or specialized instruments not available through the laptop, delivering an in person demonstration, or physically presenting to stakeholders where real time interpersonal dynamics and embodiment are central.
E0 also covers tasks where the binding constraint is access to physical environments, regulated sign off that must be done in person, or real time interpersonal dynamics such as speeches or instruction giving that cannot be meaningfully accelerated by drafting, planning, or digital coordination alone. Drafting materials for such activities may be E1, but performing the embodied activity itself remains E0 if the embodied component is the dominant share of time and effort.

E1: Direct exposure -> Label tasks E1 if using a 2026 agentic chatbot, without any additional specialized enterprise integrations beyond standard browser and workspace access, can cut the time to complete the task at equivalent quality by at least half.
E1 assumes the agent operates through general purpose capabilities: text generation, reasoning, multi step planning, browsing public or credential accessible websites, and interacting with standard productivity software. It does not assume custom APIs, deep ERP or CRM embedding, structured database level querying, or organization specific tooling built specifically for the agent.
E1 typically covers text and code generation or editing, translation, summarization and question answering over provided or web accessible documents, structured web research with citations, and routine digital administrative work that is mostly browser or workspace based such as filling forms, scheduling, building a deck from notes, cleaning a spreadsheet, or drafting and sending messages with user approval.
E1 also includes tasks that benefit substantially from structured reasoning, cognitive scaffolding, and coordination support without requiring specialized enterprise integrations. This includes breaking down ambiguous problems into structured subcomponents and actionable plans, designing one time project roadmaps, timelines, milestone plans, execution checklists, drafting project briefs, outlining negotiation strategies, generating decision matrices, performing comparative analysis across documented options, synthesizing stakeholder inputs into coherent summaries, preparing structured reports, and orchestrating communication across teams using standard digital tools. If the worker could reasonably achieve a fifty percent or greater time reduction using only the agent's standalone capabilities and manual browser interaction, the task should be labeled E1.

E2: Exposure by LLM powered applications -> Label tasks E2 if having access to the 2026 agentic chatbot alone may not reduce the time it takes to complete the task by at least half in practice, but it is easy to imagine additional software, deeper integrations, or organization specific custom tooling built on top of the agent that would reduce the time it takes by half or more.
The key distinction from E1 is that in E2 the primary bottleneck is not general reasoning, drafting, or manual browser navigation. Instead, the bottleneck lies in reliable, structured, permission aware access to proprietary systems; automated interaction with internal databases; long horizon monitoring; compliance constrained workflows; high assurance environments; or tight coupling with internal business logic. Anything that can't be done with the tool out of the box.
Examples include searching over an organization's internal knowledge and systems such as data warehouses, ticketing tools, CRM, ERP, codebases, or logs and returning precise, permission aware answers; automating multi party workflows that require approvals, compliance checks, and policy enforcement; coordinating structured procurement or contracting systems that require formal submission pipelines; or providing real time assistance embedded directly in enterprise interfaces such as a support console pulling live customer records. In these cases, the standalone agent may assist partially through drafting or reasoning, but it cannot by itself remove the core system integration bottleneck. However, with deeper enterprise integration built on top of the agent, time savings of at least half are plausible.
Examples of software built on top of the agent that may help complete worker activities include tools for a home goods company that quickly process and summarize up to date internal sales and logistics data to inform product or marketing decisions; tools that suggest live responses for customer service agents while automatically pulling relevant account context; tools for legal work that aggregate and summarize prior internal case archives while retrieving the most recent controlling authority; or systems that automatically check prescriptions against contraindication databases and regulatory rules before submission. In each case, the decisive acceleration depends on structured integration beyond what a generic browser based agent can reliably accomplish alone.

E3: Exposure given image capabilities -> Suppose you had access to both the 2026 agentic chatbot and an integrated system capable of viewing, captioning, and generating images, including reading scanned PDFs, extracting text from images with high accuracy, interpreting diagrams, and analyzing video inputs. This system cannot reliably recover extremely fine physical measurements or tolerances unless those values are explicitly provided in text.
Label tasks as E3 only when access to these image capabilities, in addition to the agentic chatbot, enables a time reduction of at least fifty percent and visual understanding is a binding constraint for that reduction. E3 should be used only when visual interpretation, visual extraction, or image generation is central to the acceleration. If comparable time savings could be achieved using text alone, the task should not be labeled E3.
Examples include extracting text from non machine readable scanned documents, structuring data from screenshots, interpreting diagrams not available in textual form, reviewing visual layouts, annotating design mockups, or creating and editing digital images and figures according to specifications. If both enterprise integration and image capabilities are relevant, assign the label corresponding to the minimal additional capability required to achieve the time reduction. E3 applies only when image understanding is the critical enabler.

Annotation examples:
Occupation: Inspectors, Testers, Sorters, Samplers, and Weighers
Task: Adjust, clean, or repair products or processing equipment to correct defects found during inspections.
Label (E0/E1/E2/E3): E0
Explanation: The model does not have access to any kind of physical embodiment, and more than half of the task described requires hands on physical adjustment, cleaning, and repair of equipment.

Occupation: Computer and Information Research Scientists
Task: Apply theoretical expertise and innovation to create or apply new technology, such as adapting principles for applying computers to new uses.
Label (E0/E1/E2/E3): E1
Explanation: The agent can provide substantial acceleration in ideation, literature discovery via browsing, technical writing, prototyping code, experiment design, structured reasoning, and critique using standalone capabilities.

Activity: Schedule dining reservations.
Label (E0/E1/E2/E3): E2
Explanation: An agent can speed up searching, comparing options, and filling booking forms on some platforms, but end to end completion often depends on fragmented access across multiple reservation systems, account logins, CAPTCHAs, and availability that may require phone confirmation. With unified reservation APIs, concierge style integrations, and reliable identity and payment handoff, it is easy to imagine a system built on top of the agent that would reduce completion time by at least half.

Activity: Negotiate purchases or contracts.
Label (E0/E1/E2/E3): E2
Explanation: The agent can draft positions, generate concession strategies, and pr opose language, but the binding constraint is often multi party adoption, trust, enforceable workflow execution, approvals, and organizational buy in. With deeper integration into contracting systems and structured approval pipelines, large time savings become more plausible.

Occupation: Allergists and Immunologists 
Task: Prescribe medication such as antihistamines, antibiotics, and nasal, oral, topical, or inhaled glucocorticosteroids.
Label (E0/E1/E2/E3): E2
Explanation: The agent can help by synthesizing patient history, drafting clinical notes, proposing differential diagnoses, and generating guideline aligned treatment options, but prescribing remains high stakes, regulated, and dependent on clinician judgment, physical examination, and formal EHR integration. With structured order entry, contraindication checking, and audit trail integration, time savings could be large, yet a human must remain in the loop for the final decision."""

log.info(f"Rubric loaded: {len(RUBRIC):,} chars")

# ── Create cached content (rubric + system instructions) ─────────────
# This avoids resending the large rubric with every request, cutting
# prompt tokens dramatically and reducing throttling pressure.

_SYSTEM_INSTRUCTIONS = f"""You are an expert labour-market analyst.
Your sole task is to classify O*NET task statements using the
2026 Agentic AI Exposure Rubric reproduced below.

{RUBRIC}

── INSTRUCTIONS ──
1. Read the occupation title(s) and task statement(s) provided by the user.
2. Apply the rubric carefully to each task.
3. When multiple tasks are provided, classify each one independently.
4. Return ONLY valid JSON (no markdown, no extra keys):
   - For a single task: {{"exposure_label": "<E0|E1|E2|E3>", "explanation": "<1-2 sentences>"}}
   - For multiple tasks: {{"results": [{{"task_index": 0, "exposure_label": "E0", "explanation": "..."}}, ...]}}

Keep each explanation concise (under 50 words).
Do not include any text before or after the JSON object."""

CACHE_TTL = "86400s"  # 1 hour; adjust as needed

try:
    cached_content = client.caches.create(
        model=MODEL_NAME,
        config={
            "display_name": "exposure-rubric-cache",
            "system_instruction": _SYSTEM_INSTRUCTIONS,
            "contents": [
                {"role": "user", "parts": [{"text": "Ready. Classify the next tasks."}]}
            ],
            "ttl": CACHE_TTL,
        },
    )
    USE_CACHE = True
    log.info(f"Cached content created: {cached_content.name}  (TTL={CACHE_TTL})")
except Exception as exc:
    cached_content = None
    USE_CACHE = False
    log.warning(f"Caching unavailable ({exc}); will send rubric inline as fallback.")

print("USE_CACHE =", USE_CACHE)
if USE_CACHE:
    print("cached_content.name =", cached_content.name)

13:24:35  INFO      Rubric loaded: 11,559 chars
13:25:19  INFO      HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/eloundou-new-scores/locations/global/cachedContents "HTTP/1.1 200 OK"
13:25:20  INFO      Cached content created: projects/1091176954194/locations/global/cachedContents/3685183438635139072  (TTL=86400s)


USE_CACHE = True
cached_content.name = projects/1091176954194/locations/global/cachedContents/3685183438635139072


## 3 · Prompt Template

In [14]:
# ── Prompt template ───────────────────────────────────────────────────
# When caching is active, _SYSTEM_INSTRUCTIONS lives in the cache and
# each request only sends the short user payload below.
# When caching is unavailable, SYSTEM_PROMPT is sent inline as before.

SYSTEM_PROMPT = _SYSTEM_INSTRUCTIONS  # fallback for non-cached path


def build_user_prompt(title: str, task: str) -> str:
    """Build the user message for a single task classification."""
    return (
        f"Occupation: {title}\n"
        f"Task: {task}\n\n"
        "Return only JSON."
    )


def build_batch_prompt(tasks: list[dict]) -> str:
    """Build the user message for a batch of task classifications.
    
    Args:
        tasks: List of dicts with keys 'title', 'task', 'task_id'
    
    Returns:
        Formatted prompt string
    """
    task_blocks = []
    for i, t in enumerate(tasks):
        task_blocks.append(
            f"Task {i}:\n"
            f"Occupation: {t['title']}\n"
            f"Task: {t['task']}"
        )
    
    prompt = "\n\n".join(task_blocks)
    prompt += "\n\nReturn only JSON with a 'results' array containing one object per task, each with 'task_index' (0-based), 'exposure_label', and 'explanation'."
    return prompt


# Quick sanity check
sample = build_user_prompt("Chief Executives",
                           "Direct or coordinate an organization's financial "
                           "or budget activities.")
print(f"USE_CACHE = {USE_CACHE}")
print(sample)

USE_CACHE = True
Occupation: Chief Executives
Task: Direct or coordinate an organization's financial or budget activities.

Return only JSON.


## 4 · Core Classification Functions

In [15]:
import re

# ── Token / cost tracking ────────────────────────────────────────────
_token_log = {"prompt_tokens": 0, "completion_tokens": 0, "calls": 0}

# NOTE: Update these if pricing changes
_COST_PER_1K_INPUT  = 0.003    # example placeholder for 3.1 Pro
_COST_PER_1K_OUTPUT = 0.015    # update to official pricing

# Retry settings for rate limits
MAX_RETRIES     = 5
INITIAL_BACKOFF = 2.0  # seconds

# ── Change 3: Adaptive global throttling ─────────────────────────────
# Instead of only per-call exponential backoff, we maintain a global
# cooldown that increases on 429 and gradually relaxes on success.
# This prevents "429 storms" where subsequent tasks keep hammering the
# quota limit.
GLOBAL_COOLDOWN  = 0.0
COOLDOWN_DECAY   = 0.95    # multiply on each success → gradual relaxation
COOLDOWN_BUMP    = 2.0     # seconds added on each 429
COOLDOWN_MAX     = 60.0    # hard cap


def _sleep_before_call():
    """Honour the global cooldown before every API call."""
    if GLOBAL_COOLDOWN > 0:
        log.debug(f"Global cooldown sleep: {GLOBAL_COOLDOWN:.2f}s")
        time.sleep(GLOBAL_COOLDOWN)


def _on_success():
    """Relax global cooldown after a successful call."""
    global GLOBAL_COOLDOWN
    GLOBAL_COOLDOWN *= COOLDOWN_DECAY


def _on_429(wait_time: float | None = None):
    """Increase global cooldown when a 429 is received."""
    global GLOBAL_COOLDOWN
    bump = wait_time if wait_time is not None else COOLDOWN_BUMP
    GLOBAL_COOLDOWN = min(COOLDOWN_MAX, GLOBAL_COOLDOWN + bump)


def _estimate_tokens(text: str) -> int:
    return max(1, len(text) // 4)


# ── JSON response schema (shared between cached and inline paths) ────
_SINGLE_TASK_SCHEMA = {
    "type": "object",
    "properties": {
        "task_index":     {"type": "integer"},
        "exposure_label": {"type": "string",
                           "enum": ["E0", "E1", "E2", "E3"]},
        "explanation":    {"type": "string"},
    },
    "required": ["task_index", "exposure_label", "explanation"],
    "additionalProperties": False,
}

_RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "results": {
            "type": "array",
            "items": _SINGLE_TASK_SCHEMA,
        },
    },
    "required": ["results"],
    "additionalProperties": False,
}


def _call(text: str) -> tuple[str, str | None]:
    """
    Low-level call to Gemini with structured JSON output.
    Returns: (raw_json_text, finish_reason)
    """
    last_exception = None

    for attempt in range(MAX_RETRIES):
        try:
            _sleep_before_call()

            config_kwargs = {
                "temperature": TEMPERATURE,
                "top_p": TOP_P,
                "max_output_tokens": MAX_OUTPUT_TOKENS,
                "response_mime_type": "application/json",
                "response_schema": _RESPONSE_SCHEMA,
            }
            if USE_CACHE and cached_content is not None:
                config_kwargs["cached_content"] = cached_content.name

            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=text,
                config=types.GenerateContentConfig(**config_kwargs),
            )
            _on_success()

            raw = (response.text or "").strip()

            finish = None
            try:
                finish = str(response.candidates[0].finish_reason)
            except (AttributeError, IndexError, TypeError):
                pass

            # Token tracking
            if hasattr(response, "usage_metadata") and response.usage_metadata:
                usage = response.usage_metadata
                _token_log["prompt_tokens"] += usage.prompt_token_count or 0
                _token_log["completion_tokens"] += usage.candidates_token_count or 0
            else:
                _token_log["prompt_tokens"] += _estimate_tokens(text)
                _token_log["completion_tokens"] += _estimate_tokens(raw)
            _token_log["calls"] += 1

            return raw, finish

        except Exception as e:
            last_exception = e
            error_str = str(e)

            if "429" in error_str or "RESOURCE_EXHAUSTED" in error_str:
                wait_time = INITIAL_BACKOFF * (2 ** attempt)
                _on_429(wait_time)
                log.warning(
                    f"Rate limit hit (attempt {attempt + 1}/{MAX_RETRIES}), "
                    f"waiting {wait_time:.1f}s..."
                )
                time.sleep(wait_time)
                continue

            raise

    raise last_exception


def _repair_json(raw: str) -> dict | None:
    """
    Attempt to repair broken JSON from the model.
    Strategy:
      1. json.loads as-is
      2. regex extraction of a JSON object
      3. suffix closing (missing } or ")
      4. label-only fallback: extract just the label
    """
    # 1. Try as-is
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass

    # 2. Regex: pull first { … }
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            pass

    # 3. Suffix closing
    trimmed = raw.rstrip()
    for suffix in ['"}', '}', '"']:
        try:
            return json.loads(trimmed + suffix)
        except json.JSONDecodeError:
            pass

    # 4. Label-only fallback
    label_match = re.search(r'"exposure_label"\s*:\s*"(E[0-3])"', raw)
    if label_match:
        return {
            "exposure_label": label_match.group(1),
            "explanation": "(auto-extracted — original JSON was malformed)",
        }

    return None


def _validate(d: dict) -> None:
    """
    Validate parsed JSON.
    Raises ValueError/KeyError if invalid.
    """
    if not isinstance(d, dict):
        raise ValueError("Model output is not a dict")
    if "exposure_label" not in d:
        raise KeyError("Missing exposure_label")
    if "explanation" not in d:
        raise KeyError("Missing explanation")
    if d["exposure_label"] not in ("E0", "E1", "E2", "E3"):
        raise ValueError(f"Invalid label: {d['exposure_label']}")


# ── Change 2: _is_truncated removed ─────────────────────────────────
# The old punctuation heuristic (`_is_truncated`) triggered false-
# positive retries on valid JSON output. We now rely exclusively on
# the API's `finish_reason == MAX_TOKENS` signal (returned by _call).


def query_model(title: str, task: str) -> dict:
    """
    Classify a single task using Gemini 3.1 Pro Preview.
    Returns dict with keys: exposure_label, explanation.

    Change 1: When caching is active, the prompt is just the short user
              payload; the rubric lives in the cache.
    Change 2: Only retries on API-indicated truncation (MAX_TOKENS),
              not on a noisy punctuation heuristic.
    """
    if USE_CACHE and cached_content is not None:
        # Cached path — only send the short user message
        prompt = build_user_prompt(title, task)
    else:
        # Fallback — inline system prompt + user message
        prompt = SYSTEM_PROMPT + "\n\n" + build_user_prompt(title, task)

    if DRY_RUN:
        return {"exposure_label": "DRY", "explanation": "dry run"}

    raw, finish = _call(prompt)

    parsed = _repair_json(raw)
    if parsed is None:
        log.warning(f"JSON repair failed: {raw[:120]}")
        return {"exposure_label": "ERROR", "explanation": raw[:300]}

    try:
        _validate(parsed)
    except (ValueError, KeyError) as e:
        log.warning(f"Validation failed: {e}")
        return {"exposure_label": "ERROR", "explanation": str(e)}

    # ── Change 2: only retry when the API explicitly says truncated ──
    if finish and "MAX_TOKENS" in finish.upper():
        log.info("API indicated truncation — retrying once with tighter constraint…")
        raw2, _ = _call(prompt)
        parsed2 = _repair_json(raw2)
        if parsed2 is not None:
            try:
                _validate(parsed2)
                parsed = parsed2  # use the retry result
            except Exception:
                pass  # keep original parsed

    return parsed


def query_model_batch(tasks: list[dict]) -> list[dict]:
    """
    Classify a batch of tasks using Gemini 3.1 Pro Preview.
    
    Args:
        tasks: List of dicts with keys 'title', 'task', 'task_id'
    
    Returns:
        List of dicts, one per task, with keys: task_id, exposure_label, explanation
    """
    if USE_CACHE and cached_content is not None:
        prompt = build_batch_prompt(tasks)
    else:
        prompt = SYSTEM_PROMPT + "\n\n" + build_batch_prompt(tasks)

    if DRY_RUN:
        return [{"task_id": t["task_id"], "exposure_label": "DRY", "explanation": "dry run"} for t in tasks]

    raw, finish = _call(prompt)

    parsed = _repair_json(raw)
    if parsed is None:
        log.warning(f"JSON repair failed for batch: {raw[:120]}")
        return [{"task_id": t["task_id"], "exposure_label": "ERROR", "explanation": raw[:300]} for t in tasks]

    # Extract results array
    if not isinstance(parsed, dict) or "results" not in parsed:
        log.warning("Batch response missing 'results' array")
        return [{"task_id": t["task_id"], "exposure_label": "ERROR", "explanation": "Invalid batch response"} for t in tasks]

    results = parsed["results"]
    if not isinstance(results, list) or len(results) != len(tasks):
        log.warning(f"Expected {len(tasks)} results, got {len(results) if isinstance(results, list) else 0}")
        # Try to salvage what we can
        output = []
        for i, t in enumerate(tasks):
            if isinstance(results, list) and i < len(results) and isinstance(results[i], dict):
                output.append({
                    "task_id": t["task_id"],
                    "exposure_label": results[i].get("exposure_label", "ERROR"),
                    "explanation": results[i].get("explanation", "Missing from batch response"),
                })
            else:
                output.append({
                    "task_id": t["task_id"],
                    "exposure_label": "ERROR",
                    "explanation": "Missing from batch response",
                })
        return output

    # Map results back to task_ids
    output = []
    for i, task_result in enumerate(results):
        if i < len(tasks):
            output.append({
                "task_id": tasks[i]["task_id"],
                "exposure_label": task_result.get("exposure_label", "ERROR"),
                "explanation": task_result.get("explanation", "No explanation provided"),
            })

    return output

In [16]:
_checkpoint_lock = threading.Lock()
_cooldown_lock   = threading.Lock()


def _run_batch(batch_tasks: list[dict]) -> list[dict]:
    """
    Worker function: classify one batch and return a list of result dicts.
    Never raises — failed batches return ERROR rows.
    """
    try:
        batch_results = query_model_batch(batch_tasks)
        return [
            {
                "Task ID":        r["task_id"],
                "exposure_label": r["exposure_label"],
                "explanation":    r.get("explanation", ""),
                "model_name":     MODEL_NAME,
                "timestamp":      datetime.now(timezone.utc).isoformat(),
            }
            for r in batch_results
        ]
    except Exception as exc:
        log.error(f"Batch failed ({[t['task_id'] for t in batch_tasks]}): {exc}")
        return [
            {
                "Task ID":        t["task_id"],
                "exposure_label": "ERROR",
                "explanation":    str(exc)[:300],
                "model_name":     MODEL_NAME,
                "timestamp":      datetime.now(timezone.utc).isoformat(),
            }
            for t in batch_tasks
        ]


def classify_tasks_parallel(
    df: pd.DataFrame,
    already_done: set = None,
) -> list[dict]:
    """
    Classify all pending tasks using ThreadPoolExecutor for batch-level parallelism.
    - PARALLEL_BATCH_WORKERS batches run concurrently.
    - Results are merged and checkpointed only from the main thread.
    - Individual batch failures mark those tasks as ERROR without killing the run.
    """
    if already_done is None:
        already_done = set()

    pending = df[~df["Task ID"].isin(already_done)]
    if MAX_TASKS is not None:
        pending = pending.head(MAX_TASKS)

    pending_list = list(pending.iterrows())
    total_tasks  = len(pending_list)

    # Build all batches up front
    all_batches: list[list[dict]] = []
    for start in range(0, total_tasks, BATCH_SIZE):
        rows = pending_list[start:start + BATCH_SIZE]
        all_batches.append([
            {"task_id": row["Task ID"], "title": str(row["Title"]), "task": str(row["Task"])}
            for _, row in rows
        ])

    n_batches = len(all_batches)
    log.info(
        f"Tasks to classify: {total_tasks:,}  "
        f"(skipping {len(already_done):,} already done)  "
        f"→ {n_batches} batches × up to {BATCH_SIZE} tasks  "
        f"| workers={PARALLEL_BATCH_WORKERS}"
    )

    results: list[dict] = []
    completed_tasks = 0

    with ThreadPoolExecutor(max_workers=PARALLEL_BATCH_WORKERS) as pool:
        future_to_batch = {pool.submit(_run_batch, b): b for b in all_batches}

        with tqdm(total=n_batches, desc="Classifying batches") as pbar:
            for future in as_completed(future_to_batch):
                batch_rows = future.result()   # never raises (handled inside _run_batch)
                results.extend(batch_rows)
                completed_tasks += len(batch_rows)
                pbar.update(1)

                # ── Checkpoint (main thread only) ──────────────────
                if completed_tasks % SAVE_EVERY_N < BATCH_SIZE and completed_tasks >= SAVE_EVERY_N:
                    _save_checkpoint(results)
                    log.info(f"Checkpoint saved  ({completed_tasks} tasks processed)")

                # ── Inter-batch cooldown ───────────────────────────
                if GLOBAL_COOLDOWN > 0:
                    time.sleep(GLOBAL_COOLDOWN)

    # Final checkpoint
    if results:
        _save_checkpoint(results)
        log.info(f"Final checkpoint saved  ({len(results)} total rows)")

    return results


def _save_checkpoint(results: list[dict]) -> None:
    """Thread-safe append + de-duplicate checkpoint write."""
    new_df = pd.DataFrame(results)
    with _checkpoint_lock:
        if CHECKPOINT_FILE.exists():
            existing = pd.read_parquet(CHECKPOINT_FILE)
            combined = pd.concat([existing, new_df], ignore_index=True)
            combined = combined.drop_duplicates(subset="Task ID", keep="last")
        else:
            combined = new_df
        combined.to_parquet(CHECKPOINT_FILE, index=False)

## 5 · Load Data & Resume from Checkpoint

In [17]:
# ── Load input tasks ─────────────────────────────────────────────────
df_tasks = pd.read_excel(INPUT_FILE)
log.info(f"Loaded {len(df_tasks):,} tasks from {INPUT_FILE}")
print(f"Columns: {list(df_tasks.columns)}")
print(f"Sample:")
df_tasks.head(3)

13:25:20  INFO      Loaded 18,796 tasks from data/Task Statements.xlsx


Columns: ['O*NET-SOC Code', 'Title', 'Task ID', 'Task', 'Task Type', 'Incumbents Responding', 'Date', 'Domain Source']
Sample:


,O*NET-SOC Code,Title,Task ID,Task,Task Type,Incumbents Responding,Date,Domain Source
0,11-1011.00,Chief Executives,8823,Direct or coordinate an organization's financi...,Core,95.0,08/2023,Incumbent
1,11-1011.00,Chief Executives,8824,"Confer with board members, organization offici...",Core,95.0,08/2023,Incumbent
2,11-1011.00,Chief Executives,8827,"Prepare budgets for approval, including those ...",Core,95.0,08/2023,Incumbent


In [18]:
# ── Resume: load checkpoint if it exists ─────────────────────────────
already_classified: set = set()

if FRESH_START and CHECKPOINT_FILE.exists():
    CHECKPOINT_FILE.unlink()
    log.info("FRESH_START=True → deleted old checkpoint. Starting from scratch.")

if CHECKPOINT_FILE.exists():
    df_ckpt = pd.read_parquet(CHECKPOINT_FILE)

    # Only count non-error rows as "done"
    done = df_ckpt[df_ckpt["exposure_label"] != "ERROR"]
    error_tasks = df_ckpt[df_ckpt["exposure_label"] == "ERROR"]
    already_classified = set(done["Task ID"].unique())

    log.info(f"Resuming — {len(already_classified):,} tasks already classified")
    if len(error_tasks) > 0:
        log.warning(f"Found {len(error_tasks):,} tasks with ERROR status — these will be RETRIED")
        print(f"\n⚠️  ERROR tasks to retry: {len(error_tasks):,}")
        print(f"   First few error Task IDs: {list(error_tasks['Task ID'].head(10))}")
else:
    log.info("No checkpoint found — starting fresh")

13:25:20  INFO      Resuming — 18,030 tasks already classified
13:25:20  WARNING   Found 766 tasks with ERROR status — these will be RETRIED



⚠️  ERROR tasks to retry: 766
   First few error Task IDs: [10623, 10617, 10620, 10629, 10633, 10624, 10631, 10627, 10625, 10628]


## 6 · Run Classification Loop

In [19]:
results = classify_tasks_parallel(df_tasks, already_done=already_classified)
log.info(f"Classification complete — {len(results):,} tasks processed this run")

13:25:20  INFO      Tasks to classify: 766  (skipping 18,030 already done)  → 77 batches × up to 10 tasks  | workers=2
13:25:20  INFO      AFC is enabled with max remote calls: 10.
13:25:20  INFO      AFC is enabled with max remote calls: 10.


Classifying batches:   0%|          | 0/77 [00:00<?, ?it/s]

13:25:37  INFO      HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/eloundou-new-scores/locations/global/publishers/google/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"
13:25:37  INFO      AFC is enabled with max remote calls: 10.
13:25:37  INFO      Checkpoint saved  (10 tasks processed)
13:25:46  INFO      HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/eloundou-new-scores/locations/global/publishers/google/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"
13:25:46  INFO      AFC is enabled with max remote calls: 10.
13:25:46  INFO      Checkpoint saved  (20 tasks processed)
13:26:02  INFO      HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/eloundou-new-scores/locations/global/publishers/google/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"
13:26:02  INFO      AFC is enabled with max remote calls: 10.
13:26:02  INFO      Checkpoint saved  (30 tasks processed)
13:26:16  

## 7 · Merge Results & Save Final Output

In [20]:
# ── Load full checkpoint (all runs combined) ─────────────────────────
df_classified = pd.read_parquet(CHECKPOINT_FILE)
log.info(f"Total entries in checkpoint: {len(df_classified):,}")

# ── Filter out ERROR entries from the checkpoint ────────────────────
error_count = len(df_classified[df_classified["exposure_label"] == "ERROR"])
if error_count > 0:
    log.warning(f"Removing {error_count:,} ERROR entries from final output")
    print(f"\n⚠️  Excluding {error_count:,} tasks that still have ERROR status")

df_classified = df_classified[df_classified["exposure_label"] != "ERROR"]
log.info(f"Successfully classified tasks: {len(df_classified):,}")

# ── Merge back into original task dataframe ──────────────────────────
df_output = df_tasks.merge(
    df_classified[["Task ID", "exposure_label", "explanation",
                   "model_name", "timestamp"]],
    on="Task ID",
    how="left",
)

# ── Save to Excel ────────────────────────────────────────────────────
df_output.to_excel(OUTPUT_FILE, index=False)
log.info(f"Saved final output: {OUTPUT_FILE}  ({len(df_output):,} rows)")

print(f"\nOutput columns: {list(df_output.columns)}")
print(f"Tasks with classifications: {df_output['exposure_label'].notna().sum():,}")
print(f"Tasks without classifications: {df_output['exposure_label'].isna().sum():,}")
df_output.head()

14:01:08  INFO      Total entries in checkpoint: 18,796
14:01:08  WARNING   Removing 10 ERROR entries from final output
14:01:08  INFO      Successfully classified tasks: 18,786



⚠️  Excluding 10 tasks that still have ERROR status


14:01:10  INFO      Saved final output: data/Task_Statements_Classified_04.xlsx  (18,796 rows)



Output columns: ['O*NET-SOC Code', 'Title', 'Task ID', 'Task', 'Task Type', 'Incumbents Responding', 'Date', 'Domain Source', 'exposure_label', 'explanation', 'model_name', 'timestamp']
Tasks with classifications: 18,786
Tasks without classifications: 10


,O*NET-SOC Code,Title,Task ID,Task,Task Type,Incumbents Responding,Date,Domain Source,exposure_label,explanation,model_name,timestamp
0,11-1011.00,Chief Executives,8823,Direct or coordinate an organization's financi...,Core,95.0,08/2023,Incumbent,E2,Coordinating financial activities requires acc...,gemini-3.1-pro-preview,2026-03-04T20:25:14.581771+00:00
1,11-1011.00,Chief Executives,8824,"Confer with board members, organization offici...",Core,95.0,08/2023,Incumbent,E0,Conferring with board members and staff involv...,gemini-3.1-pro-preview,2026-03-04T20:25:14.581788+00:00
2,11-1011.00,Chief Executives,8827,"Prepare budgets for approval, including those ...",Core,95.0,08/2023,Incumbent,E2,Preparing budgets relies heavily on internal f...,gemini-3.1-pro-preview,2026-03-04T20:25:14.581796+00:00
3,11-1011.00,Chief Executives,8826,"Direct, plan, or implement policies, objective...",Core,94.0,08/2023,Incumbent,E2,"While an agent can draft policies, implementin...",gemini-3.1-pro-preview,2026-03-04T20:25:14.581799+00:00
4,11-1011.00,Chief Executives,8834,Prepare or present reports concerning activiti...,Core,95.0,08/2023,Incumbent,E2,"Preparing detailed reports on expenses, budget...",gemini-3.1-pro-preview,2026-03-04T20:25:14.581801+00:00


## 8 · Summary Statistics & Cost Estimate

In [21]:
# ── Label distribution ───────────────────────────────────────────────
print("=" * 60)
print("EXPOSURE LABEL DISTRIBUTION (successfully classified)")
print("=" * 60)
dist = df_classified["exposure_label"].value_counts().sort_index()
for label, count in dist.items():
    pct = 100 * count / len(df_classified)
    print(f"  {label:6s}  {count:>6,}  ({pct:5.1f}%)")
print(f"  {'TOTAL':6s}  {len(df_classified):>6,}")
print(f"\nNote: ERROR entries have been excluded from this distribution.")

# ── Token / cost summary ─────────────────────────────────────────────
print(f"\n{'=' * 60}")
print("TOKEN USAGE & ESTIMATED COST")
print(f"{'=' * 60}")
print(f"  API calls          : {_token_log['calls']:,}")
print(f"  Est. prompt tokens : {_token_log['prompt_tokens']:,}")
print(f"  Est. output tokens : {_token_log['completion_tokens']:,}")

cost_in  = _token_log['prompt_tokens']     / 1000 * _COST_PER_1K_INPUT
cost_out = _token_log['completion_tokens'] / 1000 * _COST_PER_1K_OUTPUT
print(f"  Est. input cost    : ${cost_in:.4f}")
print(f"  Est. output cost   : ${cost_out:.4f}")
print(f"  Est. total cost    : ${cost_in + cost_out:.4f}")
print(f"{'=' * 60}")

EXPOSURE LABEL DISTRIBUTION (successfully classified)
  E0      10,237  ( 54.5%)
  E1       4,692  ( 25.0%)
  E2       3,336  ( 17.8%)
  E3         521  (  2.8%)
  TOTAL   18,786

Note: ERROR entries have been excluded from this distribution.

TOKEN USAGE & ESTIMATED COST
  API calls          : 76
  Est. prompt tokens : 205,645
  Est. output tokens : 43,552
  Est. input cost    : $0.6169
  Est. output cost   : $0.6533
  Est. total cost    : $1.2702


## 9 · Visualise Results

In [22]:
import plotly.express as px

dist_df = (
    df_classified["exposure_label"]
    .value_counts()
    .sort_index()
    .reset_index()
    .rename(columns={"index": "Label", "exposure_label": "Label",
                     "count": "Count"})
)

fig = px.bar(
    dist_df, x="Label", y="Count", color="Label",
    color_discrete_map={"E0": "#636efa", "E1": "#ef553b",
                        "E2": "#00cc96", "E3": "#ab63fa", "ERROR": "#ff6692"},
    title="Exposure Label Distribution across O*NET Tasks",
    text_auto=True,
)
fig.update_layout(
    template="plotly_white", width=800, height=450,
    xaxis_title="Exposure Label", yaxis_title="Number of Tasks",
)
fig.show()